In [226]:
import pandas as pd
import numpy as np

In [227]:
data = pd.read_csv("./messy_sales - Sheet1.csv")

In [228]:
print(f"dataset has {data.shape[0]} rows and {data.shape[1]} columns")

dataset has 200 rows and 8 columns


# step 1 : Checking for INFO and Missing values


In [229]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Date               200 non-null    str  
 1   Region             200 non-null    str  
 2   Product            200 non-null    str  
 3   Category           200 non-null    str  
 4   Units Sold         196 non-null    str  
 5   Unit Price         200 non-null    str  
 6   Total Sales        197 non-null    str  
 7   Profit Margin (%)  200 non-null    str  
dtypes: str(8)
memory usage: 12.6 KB


### Handling Null values


In [230]:
data.isnull().sum()

Date                 0
Region               0
Product              0
Category             0
Units Sold           4
Unit Price           0
Total Sales          3
Profit Margin (%)    0
dtype: int64

In [231]:
data[data.isnull().any(axis=1)]

,Date,Region,Product,Category,Units Sold,Unit Price,Total Sales,Profit Margin (%)
2,2/13/2024,EAST,laptop,ELEC,NaN,489.12,NaN,0.08
10,January 26 2024,South,Desktop,ELEC,NaN,$100.76,NaN,0.08
19,1/8/2024,north,Laptop,ELEC,NaN,$169.86,NaN,0.08
38,2/26/2024,north,tablet,electronics,NaN,$443.40,"11,014",0.1


### before filling NANs , lets ensure our numeric column types are correct


In [232]:
print(data[["Units Sold", "Total Sales"]].dtypes)

data_str_to_int_mask = {"twenty": 20, "thirty": 30}

data["Units Sold"] = data["Units Sold"].replace(data_str_to_int_mask)

data["Units Sold"] = pd.to_numeric(data["Units Sold"], errors="coerce")
data["Total Sales"] = pd.to_numeric(data["Total Sales"], errors="coerce")

print("*" * 50)
print("Fixed!!!")
print("*" * 50)

print(data[["Units Sold", "Total Sales"]].dtypes)

Units Sold     str
Total Sales    str
dtype: object
**************************************************
Fixed!!!
**************************************************
Units Sold     float64
Total Sales    float64
dtype: object


### filling missing data with column mean


In [233]:
data["Units Sold"] = data["Units Sold"].fillna(round(data["Units Sold"].mean(), 2))

### filling missing data with 0


In [234]:
data["Total Sales"] = data["Total Sales"].fillna(0)

In [235]:
data[["Units Sold", "Total Sales"]]

,Units Sold,Total Sales
0,32.00,0.0
1,72.00,0.0
2,34.23,0.0
3,30.00,0.0
4,47.00,0.0
...,...,...
195,20.00,23607.0
196,20.00,0.0
197,91.00,0.0
198,30.00,41246.0


# step 2 : Clean Columns names


In [236]:
data.columns.to_list()

['Date',
 'Region',
 'Product',
 'Category',
 'Units Sold',
 'Unit Price',
 'Total Sales',
 'Profit Margin (%)']

In [237]:
# detect if and columns has extra spaces
columns_with_spaces = [col for col in data.columns if col.strip() != col]

In [238]:
# clean the stripes and rename column into lower
data.columns = data.columns.str.strip().str.lower().str.replace(" ", "_")

In [239]:
data.columns.to_list()

['date',
 'region',
 'product',
 'category',
 'units_sold',
 'unit_price',
 'total_sales',
 'profit_margin_(%)']

# step 3 : fix date formats


In [240]:
data["date"]

0      January 15 2024
1            1/30/2024
2            2/13/2024
3            4/28/2024
4            2/18/2024
            ...       
195      March 15 2024
196          2/23/2024
197          2/10/2024
198          3/12/2024
199          2/19/2024
Name: date, Length: 200, dtype: str

In [241]:
data["date"] = pd.to_datetime(data["date"], format="mixed")

In [242]:
data["date"]

0     2024-01-15
1     2024-01-30
2     2024-02-13
3     2024-04-28
4     2024-02-18
         ...    
195   2024-03-15
196   2024-02-23
197   2024-02-10
198   2024-03-12
199   2024-02-19
Name: date, Length: 200, dtype: datetime64[us]

# step 4 : Standardize text columns


### transfer to lower()


In [243]:
data[["region", "product", "category"]]

,region,product,category
0,north,tablet,electronics
1,north,Tablet,Electronics
2,EAST,laptop,ELEC
3,West,Laptop,gadget
4,EAST,Phone,ELEC
...,...,...,...
195,NORTH,Tablet,ELEC
196,north,Laptop,ELEC
197,north,Desktop,Electronics
198,West,Phone,ELEC


In [244]:
for col in ["region", "product", "category"]:
    data[col] = data[col].str.lower()

In [245]:
data[["region", "product", "category"]]

,region,product,category
0,north,tablet,electronics
1,north,tablet,electronics
2,east,laptop,elec
3,west,laptop,gadget
4,east,phone,elec
...,...,...,...
195,north,tablet,elec
196,north,laptop,elec
197,north,desktop,electronics
198,west,phone,elec


### adjust categories


In [246]:
data["category"].unique().tolist()

['electronics', 'elec', 'gadget', 'gadgets']

In [247]:
categories_mask = {
    "electronics": "electronic",
    "elec": "electronic",
    "gadget": "gadget",
    "gadgets": "gadget",
}

data["category"] = data["category"].replace(categories_mask)

In [248]:
data["category"].unique().tolist()

['electronic', 'gadget']

In [249]:
data["category"].dtype

<StringDtype(storage='python', na_value=nan)>

In [250]:
data["category"] = data["category"].astype("category")

In [251]:
data["category"].dtype

CategoricalDtype(categories=['electronic', 'gadget'], ordered=False, categories_dtype=str)

# step 5 : clean currency Columns


In [252]:
data["unit_price"]

0      USD 858
1      $833.83
2       489.12
3      $733.46
4      USD 373
        ...   
195    USD 702
196     566.26
197    $123.43
198     755.84
199    $870.90
Name: unit_price, Length: 200, dtype: str

In [253]:
# data["unit_price"] = (
#     data["unit_price"]
#     .str.replace("$", "", regex=True)
#     .str.replace("USD", "", regex=False)
#     .str.strip()
# )

data["unit_price"] = (
    data["unit_price"].str.replace("[$,USD]", "", regex=True).str.strip()
)


data["unit_price"] = pd.to_numeric(data["unit_price"], errors="coerce")

In [254]:
data["unit_price"]

0      858.00
1      833.83
2      489.12
3      733.46
4      373.00
        ...  
195    702.00
196    566.26
197    123.43
198    755.84
199    870.90
Name: unit_price, Length: 200, dtype: float64

# step 6 : fix Profit Margin (%)


In [255]:
data["profit_margin_(%)"]

0             10%
1      15 percent
2            0.08
3             0.1
4             0.1
          ...    
195           10%
196    15 percent
197           0.1
198          0.08
199           12%
Name: profit_margin_(%), Length: 200, dtype: str

In [256]:
data["profit_margin_(%)"] = data["profit_margin_(%)"].str.extract(r"(\d+\.?\d*)")

In [260]:
data["profit_margin_(%)"] = pd.to_numeric(data["profit_margin_(%)"], errors="coerce")

In [258]:
data["profit_margin_(%)"] = np.where(
    data["profit_margin_(%)"] < 1,
    data["profit_margin_(%)"] * 100,
    data["profit_margin_(%)"],
)

In [259]:
data["profit_margin_(%)"]

0      10.0
1      15.0
2       8.0
3      10.0
4      10.0
       ... 
195    10.0
196    15.0
197    10.0
198     8.0
199    12.0
Name: profit_margin_(%), Length: 200, dtype: float64

# step 7 : Drop rows with missing key data


In [265]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               200 non-null    datetime64[us]
 1   region             200 non-null    str           
 2   product            200 non-null    str           
 3   category           200 non-null    category      
 4   units_sold         200 non-null    float64       
 5   unit_price         200 non-null    float64       
 6   total_sales        200 non-null    float64       
 7   profit_margin_(%)  200 non-null    float64       
dtypes: category(1), datetime64[us](1), float64(4), str(2)
memory usage: 11.3 KB


In [ ]:
data.dropna(
    subset=["date", "total_sales", "region"],
    inplace=True,
)

,index,date,region,product,category,units_sold,unit_price,total_sales,profit_margin_(%)
0,0,2024-01-15,north,tablet,electronic,32.00,858.00,0.0,10.0
1,1,2024-01-30,north,tablet,electronic,72.00,833.83,0.0,15.0
2,2,2024-02-13,east,laptop,electronic,34.23,489.12,0.0,8.0
3,3,2024-04-28,west,laptop,gadget,30.00,733.46,0.0,10.0
4,4,2024-02-18,east,phone,electronic,47.00,373.00,0.0,10.0
...,...,...,...,...,...,...,...,...,...
195,195,2024-03-15,north,tablet,electronic,20.00,702.00,23607.0,10.0
196,196,2024-02-23,north,laptop,electronic,20.00,566.26,0.0,15.0
197,197,2024-02-10,north,desktop,electronic,91.00,123.43,0.0,10.0
198,198,2024-03-12,west,phone,electronic,30.00,755.84,41246.0,8.0


In [274]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               200 non-null    datetime64[us]
 1   region             200 non-null    str           
 2   product            200 non-null    str           
 3   category           200 non-null    category      
 4   units_sold         200 non-null    float64       
 5   unit_price         200 non-null    float64       
 6   total_sales        200 non-null    float64       
 7   profit_margin_(%)  200 non-null    float64       
dtypes: category(1), datetime64[us](1), float64(4), str(2)
memory usage: 11.3 KB


In [273]:
data.head(15)

,date,region,product,category,units_sold,unit_price,total_sales,profit_margin_(%)
0,2024-01-15,north,tablet,electronic,32.00,858.00,0.0,10.0
1,2024-01-30,north,tablet,electronic,72.00,833.83,0.0,15.0
2,2024-02-13,east,laptop,electronic,34.23,489.12,0.0,8.0
3,2024-04-28,west,laptop,gadget,30.00,733.46,0.0,10.0
4,2024-02-18,east,phone,electronic,47.00,373.00,0.0,10.0
5,2024-04-03,south,laptop,gadget,20.00,670.28,3103.0,8.0
6,2024-01-28,north,tablet,electronic,30.00,758.58,0.0,8.0
7,2024-03-15,west,tablet,gadget,47.00,241.65,0.0,12.0
8,2024-01-09,west,phone,gadget,30.00,796.00,0.0,8.0
9,2024-02-07,west,laptop,gadget,30.00,996.92,0.0,8.0
